# Grounding Inspector — Validation Notebook

**Goal:** Validate the decompose-then-verify pipeline against a public hallucination benchmark (RAGTruth).
Report the DETECTOR scorecard: recall with Clopper-Pearson CI, balanced accuracy, and Cohen's κ.

**Key framing:** An honest 85 beats a fake 100.
The numbers below certify the detector on the *benchmark distribution* — not on travel-insurance PDSs.
The domain-transfer caveat is load-bearing; do not elide it.

## Methodology

Pipeline follows the field-standard **decompose-then-verify** pattern (FActScore lineage).

### Decomposition
Split the AI output into atomic, independently verifiable statements via a **fixed, versioned LLM prompt** (v1).
Sensitivity caveat: FActScore-style scores shift with different decomposers.
The prompt is fixed and versioned (`DECOMPOSE_PROMPT` in `grounding/decompose.py`).

### Verification (full-document, no retrieval gate)
Primary verifier: **MiniCheck** (MiniCheck-FT5, flan-t5-large, 770M parameters).
Each claim is scored against the *whole document* (chunk + max-pool).
Crucially, we do **not** pre-select top-k chunks — retrieval recall would silently cap detection recall.

### Benchmark
**RAGTruth** (~18k examples, response- and span-level hallucination labels).
We use a stratified sample of n=300 to keep the run time within one deep-work block.
We do not target exact reproduction of published numbers — label-mapping and version drift can explain gaps.

### Recall
"Recall" = fraction of genuinely hallucinated/unsupported claims caught by the detector.
A false negative is a *missed hallucination* — the dangerous error.
Reported with a **Clopper-Pearson 95% CI** sized by the number of positive examples in the sample.

In [ ]:
import sys
sys.path.insert(0, '..')   # make grounding package importable from engine/

from grounding.validate import run_sample, summarise, map_ragtruth_label
from grounding.metrics import recall_with_ci
import json

## Run the benchmark sample

This cell downloads the RAGTruth dataset (first run only) and runs MiniCheck over n=300 examples.
**Expected runtime: ~15-20 minutes on CPU (MacBook Pro M-series).**
MiniCheck model weights (~300 MB) are cached in `engine/ckpts/` after first download.

> Time-box rule: if this overruns one deep-work block, ship the notebook with `results = None`
> and note 'validation in progress'. The demo artifact stands without it.

In [ ]:
# Run once; results take ~15-20 min on CPU
# Set results = None to skip and use the placeholder scorecard below
results = None

# Uncomment to run:
# results = run_sample(n=300, seed=0)
# print(json.dumps(results, indent=2))

## DETECTOR Scorecard

Fill in from `results` above once the run completes, or use the placeholder to document the methodology.

In [ ]:
# Replace with results from run_sample() after execution
scorecard = results or {
    "recall": None,
    "recall_ci": [None, None],
    "false_negatives": None,
    "n_positive": None,
    "balanced_accuracy": None,
    "cohen_kappa": None,
    "note": "Validation run pending — placeholder only"
}

print("=== DETECTOR SCORECARD ===")
print(f"Validated on: RAGTruth (sampled, n=300, seed=0)")
print(f"Recall:            {scorecard['recall']}")
print(f"  95% CI:          {scorecard['recall_ci']}")
print(f"  False negatives: {scorecard['false_negatives']} / {scorecard['n_positive']} positives")
print(f"Balanced accuracy: {scorecard['balanced_accuracy']}")
print(f"Cohen's kappa:     {scorecard['cohen_kappa']}")

## Honest-Ceiling Narrative

### What the numbers certify

The scorecard above measures the detector's recall on the **RAGTruth distribution** —
primarily summarisation and QA tasks, with human-annotated hallucination spans.
The recall CI quantifies how often MiniCheck catches a genuinely hallucinated claim in that setting.

### What the numbers do not certify

**Domain-transfer caveat (load-bearing — do not remove):**

RAGTruth is summarisation/QA, not insurance-PDS triage.
The benchmark numbers say nothing about the detector's behaviour on:

- **Numeric entailment** — e.g., "covered up to \$5,000" vs "sub-limit of \$1,000 per item".
  Quantitative overstatement is a known NLI/MiniCheck weak spot.
- **Legal/regulatory exclusion language** — e.g., pandemic exclusions phrased in insurance-specific terms.
- **Negation in dense lists** — e.g., "X is covered *unless* Y" in a 30-item exclusion table.

The 2-3 synthetic + 2 real-PDS fixtures are an **out-of-distribution illustration**
with author-asserted ground truth, not measured recall.
Closing the domain gap would require a labelled PDS evaluation set — deliberately not built here.

### Decomposer sensitivity

The pipeline score is sensitive to the decomposer prompt.
A different prompt yields a different sub-claim set and shifts the groundedness score,
even if the verifier (MiniCheck) is held constant.
The prompt is fixed at v1 (`DECOMPOSE_PROMPT` in `grounding/decompose.py`).
Any prompt change constitutes a new pipeline version and invalidates prior comparisons.

### The framing

> **An honest 85 beats a fake 100.**
>
> A recall of 0.85 with a tight CI and an honest domain-transfer caveat is more useful
> than a claim of perfect detection with no validation.
> The CI tells you what you can rely on; the caveat tells you where you cannot.

## Updating Fixture Scorecards

Once `results` is populated, update every fixture's `scorecard` block:

```python
import json, pathlib

FIXTURES_DIR = pathlib.Path('..') / 'fixtures'
SCORECARD = {
    "recall": results["recall"],
    "recall_ci": results["recall_ci"],
    "false_negatives": results["false_negatives"],
    "n_positive": results["n_positive"],
    "citation_precision": None,
    "cohen_kappa": results["cohen_kappa"],
    "validated_on": "RAGTruth (sampled, n=300, seed=0)",
    "domain_note": "benchmark distribution; NOT measured on PDS",
}

for path in sorted(FIXTURES_DIR.glob('*.json')):
    fx = json.loads(path.read_text())
    fx['scorecard'] = SCORECARD
    path.write_text(json.dumps(fx, indent=2))
    print(f'Updated {path.name}')
```